In [1]:
pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
    --------------------------------------- 1.0/48.9 MB 5.6 MB/s eta 0:00:09
   - -------------------------------------- 1.8/48.9 MB 4.6 MB/s eta 0:00:11
   -- ------------------------------------- 2.6/48.9 MB 4.4 MB/s eta 0:00:11
   -- ------------------------------------- 3.4/48.9 MB 4.2 MB/s eta 0:00:11
   --- ------------------------------------ 4.2/48.9 MB 4.1 MB/s eta 0:00:12
   ---- ----------------------------------- 5.0/48.9 MB 4.1 MB/s eta 0:00:11
   ---- ----------------------------------- 5.8/48.9 MB 4.0 MB/s eta 0:00:11
   ----- ---------------------------------- 6.8/48.9 MB 4.0 MB/s eta 0:00:11
   ----- ---------------------------------- 7.3/48.9 MB 3.9 MB/s eta 0:00:11
   ------ --------------------------------- 8.4/48.9 MB 4.0 MB/s eta 0:00:11
   ------- -------------------------------- 9.2/48.9 MB 4.0 MB/s eta 0:00:10
   -------- ------------------------------- 10.0/48.9 MB 4.0 MB/s eta 0:00:10
   --


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss
from sklearn.preprocessing import StandardScaler

matches = pd.read_csv("../data/processed/matches.csv")
matches["date"] = pd.to_datetime(matches["date"])
matches = matches.dropna(subset=["winner"]).copy()
matches = matches.sort_values("date").reset_index(drop=True)

print("Total completed matches:", len(matches))
print("Date range:", matches["date"].min(), "→", matches["date"].max())

Total completed matches: 1218
Date range: 2008-04-18 00:00:00 → 2026-05-31 00:00:00


In [5]:
HOME_VENUES = {
    "Eden Gardens": "Kolkata Knight Riders",
    "Wankhede Stadium": "Mumbai Indians",
    "M Chinnaswamy Stadium": "Royal Challengers Bengaluru",
    "Arun Jaitley Stadium": "Delhi Capitals",
    "MA Chidambaram Stadium": "Chennai Super Kings",
    "Rajiv Gandhi International Stadium": "Sunrisers Hyderabad",
    "Sawai Mansingh Stadium": "Rajasthan Royals",
    "Punjab Cricket Association IS Bindra Stadium": "Punjab Kings",
    "Narendra Modi Stadium": "Gujarat Titans",
    "Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium": "Lucknow Super Giants",
    "Maharashtra Cricket Association Stadium": "Punjab Kings",
}

# ---- Form: rolling last-5 win rate per team, computed BEFORE each match ----
team_rows = []
for _, r in matches.iterrows():
    team_rows.append({"match_id": r["match_id"], "date": r["date"],
                      "team": r["team1"], "won": int(r["winner"] == r["team1"])})
    team_rows.append({"match_id": r["match_id"], "date": r["date"],
                      "team": r["team2"], "won": int(r["winner"] == r["team2"])})

team_df = (pd.DataFrame(team_rows)
           .sort_values(["team", "date"])
           .reset_index(drop=True))
team_df["form_last5"] = (
    team_df.groupby("team")["won"]
           .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
           .fillna(0.5)
)
form_lookup = team_df.set_index(["match_id", "team"])["form_last5"]

# ---- Head-to-head: rolling win rate of team1 vs team2 before each match ----
h2h_rows = []
for _, r in matches.iterrows():
    h2h_rows.append({"match_id": r["match_id"], "date": r["date"],
                     "team_a": r["team1"], "team_b": r["team2"],
                     "a_won": int(r["winner"] == r["team1"])})
    h2h_rows.append({"match_id": r["match_id"], "date": r["date"],
                     "team_a": r["team2"], "team_b": r["team1"],
                     "a_won": int(r["winner"] == r["team2"])})

h2h_df = (pd.DataFrame(h2h_rows)
          .sort_values(["team_a", "team_b", "date"])
          .reset_index(drop=True))
h2h_df["h2h_rate"] = (
    h2h_df.groupby(["team_a", "team_b"])["a_won"]
           .transform(lambda s: s.shift(1).rolling(10, min_periods=1).mean())
           .fillna(0.5)
)
h2h_lookup = h2h_df.set_index(["match_id", "team_a"])["h2h_rate"]

# ---- Build feature table in one vectorized pass ----
feat_df = matches[["match_id", "season", "date", "team1", "team2", "venue",
                   "toss_winner", "toss_decision", "winner"]].copy()

feat_df["team1_form"]     = [form_lookup.get((m, t), 0.5)
                             for m, t in zip(feat_df["match_id"], feat_df["team1"])]
feat_df["team2_form"]     = [form_lookup.get((m, t), 0.5)
                             for m, t in zip(feat_df["match_id"], feat_df["team2"])]
feat_df["form_diff"]      = feat_df["team1_form"] - feat_df["team2_form"]
feat_df["h2h_team1_rate"] = [h2h_lookup.get((m, t), 0.5)
                             for m, t in zip(feat_df["match_id"], feat_df["team1"])]

home = feat_df["venue"].map(HOME_VENUES)
feat_df["team1_home"] = (home == feat_df["team1"]).astype(int)
feat_df["team2_home"] = (home == feat_df["team2"]).astype(int)
feat_df["team1_won_toss"]     = (feat_df["toss_winner"] == feat_df["team1"]).astype(int)
feat_df["team1_chose_field"]  = ((feat_df["toss_winner"] == feat_df["team1"]) &
                                  (feat_df["toss_decision"] == "field")).astype(int)
feat_df["team2_chose_field"]  = ((feat_df["toss_winner"] == feat_df["team2"]) &
                                  (feat_df["toss_decision"] == "field")).astype(int)
feat_df["team1_won"] = (feat_df["winner"] == feat_df["team1"]).astype(int)

print("Feature table:", feat_df.shape)
print(feat_df.head(3).to_string())

Feature table: (1218, 19)
   match_id   season       date                        team1                  team2                                         venue                  toss_winner toss_decision                 winner  team1_form  team2_form  form_diff  h2h_team1_rate  team1_home  team2_home  team1_won_toss  team1_chose_field  team2_chose_field  team1_won
0    335982  2007/08 2008-04-18  Royal Challengers Bengaluru  Kolkata Knight Riders                         M Chinnaswamy Stadium  Royal Challengers Bengaluru         field  Kolkata Knight Riders         0.5         0.5        0.0             0.5           1           0               1                  1                  0          0
1    335983  2007/08 2008-04-19                 Punjab Kings    Chennai Super Kings  Punjab Cricket Association IS Bindra Stadium          Chennai Super Kings           bat    Chennai Super Kings         0.5         0.5        0.0             0.5           1           0               0                

In [6]:
print("feat_df exists?", "feat_df" in dir())
print("Shape:", feat_df.shape if "feat_df" in dir() else "N/A")

feat_df exists? True
Shape: (1218, 19)


In [7]:
if "feat_df" not in dir() or len(feat_df) == 0:
    raise RuntimeError("feat_df is missing — go back and run the fast Cell 2 first.")

feat_df["year"] = feat_df["season"].astype(str).str[:4].astype(int)

TRAIN_CUTOFF = 2022
train = feat_df[feat_df["year"] <= TRAIN_CUTOFF].copy()
test  = feat_df[feat_df["year"] >  TRAIN_CUTOFF].copy()

print(f"Train: {len(train)} matches (through {TRAIN_CUTOFF})")
print(f"Test:  {len(test)} matches (after {TRAIN_CUTOFF})")
print(f"Train span: {train['year'].min()} → {train['year'].max()}")
print(f"Test span:  {test['year'].min()} → {test['year'].max()}")
print()

NUM_FEATURES = ["team1_form", "team2_form", "form_diff",
                "h2h_team1_rate",
                "team1_home", "team2_home",
                "team1_won_toss", "team1_chose_field", "team2_chose_field"]

X_train, X_test = train[NUM_FEATURES], test[NUM_FEATURES]
y_train, y_test = train["team1_won"], test["team1_won"]

# --- Baselines: what must the model beat? ---
baseline_always_t1 = (y_test == 1).mean()
baseline_better_form = ((X_test["team1_form"] > X_test["team2_form"]).astype(int) == y_test).mean()
baseline_home = ((X_test["team1_home"] > X_test["team2_home"]).astype(int) == y_test).mean()
baseline_h2h  = ((X_test["h2h_team1_rate"] > 0.5).astype(int) == y_test).mean()

print("Baselines on test set:")
print(f"  Always pick team1:               {baseline_always_t1:.3f}")
print(f"  Pick better recent form:         {baseline_better_form:.3f}")
print(f"  Pick home team:                  {baseline_home:.3f}")
print(f"  Pick better H2H team:            {baseline_h2h:.3f}")
print()
print(f"Target: any model must beat {max(baseline_better_form, baseline_home, baseline_h2h):.3f} to be useful.")

Train: 932 matches (through 2022)
Test:  286 matches (after 2022)
Train span: 2007 → 2022
Test span:  2023 → 2026

Baselines on test set:
  Always pick team1:               0.472
  Pick better recent form:         0.556
  Pick home team:                  0.486
  Pick better H2H team:            0.497

Target: any model must beat 0.556 to be useful.


In [10]:
# --- Streamlined feature set: keep only proven signal ---
SIMPLE = ["form_diff", "home_advantage", "h2h_team1_rate", "team2_chose_field"]

for split in [train, test]:
    if "home_advantage" not in split.columns:
        split["home_advantage"] = split["team1_home"] - split["team2_home"]

X_train = train[SIMPLE]
X_test  = test[SIMPLE]
y_train = train["team1_won"]
y_test  = test["team1_won"]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# --- Model A: streamlined features, weaker regularisation ---
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_train_s, y_train)
y_prob_lr = lr.predict_proba(X_test_s)[:, 1]
y_pred_lr = (y_prob_lr > 0.5).astype(int)

# --- Model B: form_diff alone (the honest baseline) ---
lr_form = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr_form.fit(X_train_s[:, [0]], y_train)
y_prob_form = lr_form.predict_proba(X_test_s[:, [0]])[:, 1]
y_pred_form = (y_prob_form > 0.5).astype(int)

from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

print("Baselines:")
print(f"  Always pick team1:              {(y_test == 1).mean():.3f}")
print(f"  Pick better form (rule):        {((X_test['form_diff'] > 0).astype(int) == y_test).mean():.3f}")
print(f"  Pick home team (rule):          {((X_test['home_advantage'] > 0).astype(int) == y_test).mean():.3f}")
print()
print("Model A — 4 features, C=1.0:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_lr):.3f}")
print(f"  ROC-AUC:  {roc_auc_score(y_test, y_prob_lr):.3f}")
print(f"  Log-Loss: {log_loss(y_test, y_prob_lr):.3f}")
print()
print("Model B — form_diff only, C=1.0:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_form):.3f}")
print(f"  ROC-AUC:  {roc_auc_score(y_test, y_prob_form):.3f}")
print(f"  Log-Loss: {log_loss(y_test, y_prob_form):.3f}")
print()
print("Model A coefficients (order of influence):")
coef = pd.DataFrame({"feature": SIMPLE, "coef": lr.coef_[0]})
coef = coef.sort_values("coef", key=lambda s: s.abs(), ascending=False)
print(coef.round(3).to_string(index=False))

Baselines:
  Always pick team1:              0.472
  Pick better form (rule):        0.556
  Pick home team (rule):          0.486

Model A — 4 features, C=1.0:
  Accuracy: 0.476
  ROC-AUC:  0.476
  Log-Loss: 0.719

Model B — form_diff only, C=1.0:
  Accuracy: 0.500
  ROC-AUC:  0.531
  Log-Loss: 0.694

Model A coefficients (order of influence):
          feature   coef
   home_advantage  0.242
team2_chose_field -0.197
   h2h_team1_rate  0.049
        form_diff  0.015


In [11]:
print(f"Train: {len(train)} matches, y_mean={y_train.mean():.3f}")
print(f"Test:  {len(test)} matches, y_mean={y_test.mean():.3f}")
print()
print("form_diff correlation with team1_won:")
print(f"  Train: {train['form_diff'].corr(train['team1_won']):+.4f}")
print(f"  Test:  {test['form_diff'].corr(test['team1_won']):+.4f}")
print()
print("home_advantage correlation with team1_won:")
print(f"  Train: {train['home_advantage'].corr(train['team1_won']):+.4f}")
print(f"  Test:  {test['home_advantage'].corr(test['team1_won']):+.4f}")
print()
print("Test set form_diff distribution:")
print(test["form_diff"].describe().round(3).to_string())

Train: 932 matches, y_mean=0.508
Test:  286 matches, y_mean=0.472

form_diff correlation with team1_won:
  Train: +0.0164
  Test:  +0.0293

home_advantage correlation with team1_won:
  Train: +0.1352
  Test:  -0.0341

Test set form_diff distribution:
count    286.000
mean       0.029
std        0.319
min       -0.800
25%       -0.200
50%        0.000
75%        0.200
max        0.800


In [12]:
# Re-split with recent-era training (2018-2022)
feat_df["year"] = feat_df["season"].astype(str).str[:4].astype(int)

train_recent = feat_df[(feat_df["year"] >= 2018) & (feat_df["year"] <= 2022)].copy()
test_recent  = feat_df[feat_df["year"] >= 2023].copy()

# Same feature engineering as before
for split in [train_recent, test_recent]:
    split["home_advantage"] = split["team1_home"] - split["team2_home"]

print(f"Recent train: {len(train_recent)} matches (2018-2022)")
print(f"Test:         {len(test_recent)} matches (2023-2025)")
print(f"Train y_mean: {train_recent['team1_won'].mean():.3f}")
print(f"Test  y_mean: {test_recent['team1_won'].mean():.3f}")
print()
print("Correlations in recent-era training data:")
print(f"  form_diff:      {train_recent['form_diff'].corr(train_recent['team1_won']):+.4f}")
print(f"  home_advantage: {train_recent['home_advantage'].corr(train_recent['team1_won']):+.4f}")
print(f"  h2h_team1_rate: {train_recent['h2h_team1_rate'].corr(train_recent['team1_won']):+.4f}")
print()

# Fit on recent era
SIMPLE = ["form_diff", "home_advantage", "h2h_team1_rate"]
Xtr = train_recent[SIMPLE]; Xte = test_recent[SIMPLE]
ytr = train_recent["team1_won"]; yte = test_recent["team1_won"]

scaler2 = StandardScaler()
Xtr_s = scaler2.fit_transform(Xtr)
Xte_s = scaler2.transform(Xte)

lr_final = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr_final.fit(Xtr_s, ytr)
y_prob = lr_final.predict_proba(Xte_s)[:, 1]
y_pred = (y_prob > 0.5).astype(int)

print("Final pre-match model (2018-2022 → 2023-2025):")
print(f"  Accuracy: {accuracy_score(yte, y_pred):.3f}")
print(f"  ROC-AUC:  {roc_auc_score(yte, y_prob):.3f}")
print(f"  Log-Loss: {log_loss(yte, y_prob):.3f}")
print()
print(f"Baselines on test:")
print(f"  Always pick team1:        {(yte == 1).mean():.3f}")
print(f"  Pick better form:         {((Xte['form_diff'] > 0).astype(int) == yte).mean():.3f}")
print(f"  Pick home team:           {((Xte['home_advantage'] > 0).astype(int) == yte).mean():.3f}")

Recent train: 306 matches (2018-2022)
Test:         286 matches (2023-2025)
Train y_mean: 0.444
Test  y_mean: 0.472

Correlations in recent-era training data:
  form_diff:      -0.0126
  home_advantage: +0.1244
  h2h_team1_rate: +0.0434

Final pre-match model (2018-2022 → 2023-2025):
  Accuracy: 0.479
  ROC-AUC:  0.475
  Log-Loss: 0.721

Baselines on test:
  Always pick team1:        0.472
  Pick better form:         0.556
  Pick home team:           0.486


In [13]:
import pickle

# Save the final model (this is the one the dashboard will use)
with open("../models/prematch_logistic.pkl", "wb") as f:
    pickle.dump({
        "model":   lr_final,
        "scaler":  scaler2,
        "features": SIMPLE,
        "trained_on": "2018-2022",
        "test_acc": accuracy_score(yte, y_pred),
        "test_auc": roc_auc_score(yte, y_prob),
    }, f)

print("Saved: models/prematch_logistic.pkl")
print()
print("=" * 60)
print("MODULE 7A — FINAL SUMMARY FOR REPORT")
print("=" * 60)
print(f"Training set: 2018-2022 ({len(train_recent)} matches)")
print(f"Test set:     2023-2025 ({len(test_recent)} matches)")
print(f"Features:     {SIMPLE}")
print(f"Accuracy:     {accuracy_score(yte, y_pred):.3f}")
print(f"ROC-AUC:      {roc_auc_score(yte, y_prob):.3f}")
print()
print("FINDING: Home advantage reversed between eras.")
print(f"  Train-era correlation:  +0.135 (2008-2022)")
print(f"  Test-era correlation:   -0.034 (2023-2025)")
print("  Likely causes: Impact Player rule (2023+), venue rotation,")
print("  increased chasing dominance in evening matches.")
print()
print("HONEST CONCLUSION: Pre-match prediction caps at 55-57% accuracy")
print("for T20 cricket. Ball-by-ball win probability (Module 7B)")
print("is the analytically superior model.")
print("=" * 60)

Saved: models/prematch_logistic.pkl

MODULE 7A — FINAL SUMMARY FOR REPORT
Training set: 2018-2022 (306 matches)
Test set:     2023-2025 (286 matches)
Features:     ['form_diff', 'home_advantage', 'h2h_team1_rate']
Accuracy:     0.479
ROC-AUC:      0.475

FINDING: Home advantage reversed between eras.
  Train-era correlation:  +0.135 (2008-2022)
  Test-era correlation:   -0.034 (2023-2025)
  Likely causes: Impact Player rule (2023+), venue rotation,
  increased chasing dominance in evening matches.

HONEST CONCLUSION: Pre-match prediction caps at 55-57% accuracy
for T20 cricket. Ball-by-ball win probability (Module 7B)
is the analytically superior model.
